In [1]:
import hail as hl
hl.init()

Loading BokehJS ...

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
Running on Apache Spark version 3.5.5
SparkUI available at http://10.1.13.82:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.133-4c60fddb171a
LOGGING: writing to /Users/enason/Documents/vsm_family/filter_files_v1/hail-20260416-0955-0.2.133-4c60fddb171a.log
2026-04-16 11:12:16.240 Hail: INFO: Coerced sorted dataset
2026-04-16 11:12:16.507 Hail: INFO: Coerced dataset with out-of-order partitions.
2026-04-16 11:13:15.144 Hail: INFO: merging 455 files totalling 12.6M...
2026-04-16 11:13:15.342 Hail: INFO: while writing:
    mutation_rate_chr22.tsv.bgz
  merge time: 197.801ms
2026-04-16 12:17:13.398 Hail: INFO: Coerced sorted dataset
2026-04-16 12:17:13.596 Hail: INFO: Coerced dataset with out-of-order partitions.
2026-04-

In [2]:
#mut = hl.read_table("gs://missense-scoring/mutation/gnomad.v4.0.mutation_rate.ht")
ht = hl.read_table('gs://missense-scoring/mutation/everything_raw.ht')

In [3]:
ht.describe()

----------------------------------------
Global fields:
    'grange': array<int32> 
    'vep_help': str 
    'vep_config': str 
    'version': str 
----------------------------------------
Row fields:
    'idx': int32 
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'a_index': int32 
    'was_split': bool 
    'old_locus': locus<GRCh38> 
    'old_alleles': array<str> 
    'context': str 
    'coverage_mean': float64 
    'coverage_10': float32 
    'coverage_20': float32 
    'methyl_mean': str 
    'vep': struct {
        allele_string: str, 
        colocated_variants: array<struct {
            allele_string: str, 
            clin_sig: array<str>, 
            clin_sig_allele: str, 
            end: int32, 
            id: str, 
            phenotype_or_disease: int32, 
            pubmed: array<int32>, 
            somatic: int32, 
            start: int32, 
            strand: int32
        }>, 
        context: str, 
        end: int32, 
        id: str, 
        inpu

In [4]:
ht = ht.annotate(
    chrom = ht.locus.contig,
    pos   = ht.locus.position,
    ref   = ht.alleles[0],
    alt   = ht.alleles[1]
)
ht = ht.key_by("chrom", "pos", "ref", "alt")
ht = ht.select("mu_snp")

In [5]:
ht_c = ht.filter(ht.chrom == "chr22")
ht_c.export('mutation_rate_chr22.tsv.bgz')

SLF4J: Failed to load class "org.slf4j.impl.StaticMDCBinder".
SLF4J: Defaulting to no-operation MDCAdapter implementation.
SLF4J: See http://www.slf4j.org/codes.html#no_static_mdc_binder for further details.


In [6]:
ht.export('mutation_rate.tsv.bgz')